## 08 Middleware

Middleware provides a way to more tightly control what happens inside the agent. Middleware is useful for the following:
* Tracking agent behavior with logging, analytics, and debugging.
* Transforming prompts, tool selection, and output formatting.
* Adding retries, fallbacks, and early termination logic.
* Applying rate limits, guardrails, and PII detection.

### The core agent loop
The core agent loop involves calling a model, letting it choose tools to execute, and then finishing when it calls no more tools:

<div align="center">
<img src="images/01_core_agent_loop.avif" width="250" heigh="100" alt="Core Agent Loop"/>
</div>

Middleware exposes hooks before and after each of those steps:

<div align="center">
<img src="images/08_middleware_final.avif" width="250" heigh="100" alt="Middleware"/>
</div>

LangChain supports **pre-built** as well as **custom** middleware. In this notebook we'll see examples of each type.

### Pre-built Middleware
LangChain provides prebuilt middleware for common use cases. Each middleware is production-ready and configurable for your specific needs. Please refer to [pre-built middleware](https://docs.langchain.com/oss/python/langchain/middleware/built-in) page for more details.

In  the following example, we'll show how to apply a PII filter to blank-out PII information such as address, phone, email etc.

Overview of this example:
1. We'll query our `Chinook` database, specifically for details of our customers, which will include PII information such as customer name, address, email and phone-number.
2. Our objective is to redact this information before or after the LLM sees the information - our ultimate objective is to hide this PII information from end-user (he/she should not see this info!)

To speed up querying (or SQL generation), we will feed the database scheme to the agent. For this we'll employ another Middleware trick to modify the system prompt.

In [1]:
from dotenv import load_dotenv
from rich.console import Console
from rich.markdown import Markdown
from dataclasses import dataclass

import langchain
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain.tools import tool
from langchain_community.utilities import SQLDatabase

print(f"Using langchain version: {langchain.__version__}")

load_dotenv(override=True)
console = Console()

Using langchain version: 1.2.14


**Step 1:**

As a first step, let's load the `Chinook` database and it's schema. The schema will help our agent generate the query faster. 

Next we define the runtime context for the LLM to hold the database connection as well as the schema information. We also define the structure of the output we expect from our agent - the details of the customer we want to extract from our database (a `Customer` class derived from Pydantic `BaseModel`). Since our query could return multiple rows, we define the response format as a Pydantic `BaseModel` derived class that holds a `List` of `Customer` objects. 

In [2]:
from langchain_community.utilities import SQLDatabase
from typing import TypedDict, Optional, List
from pydantic import BaseModel

# connect to our database -> in path db/chinook.db
db = SQLDatabase.from_uri("sqlite:///db/chinook.db")
schema = db.get_table_info()


# define our runtime context - we'll pass in the schema this time
class RuntimeContext(TypedDict):
    db: SQLDatabase
    db_schema: str


# this class defines the details (attributes) of the customer we want to extract
# CustomerId, FirstName, LastName, Address, City, State, Country, Phone, Email
class Customer(BaseModel):
    customerId: str
    firstName: str
    lastName: str
    address: Optional[str] = None  # will be masked out
    city: Optional[str] = None
    state: Optional[str] = None
    country: Optional[str] = None
    phone: Optional[str] = None  # will be masked out
    email: Optional[str] = None  # will be masked out


# Our query could return 1 or more rows, so the output format from our
# agent should be a List of Customer rows
class Customers(BaseModel):
    customers: List[Customer]

**Step 2**

Next we define the tool (the usual `execute_sql` we have used before), which will execute SQL generated by the agent and return the result as a string. As in previous notebooks, this function gets an instance of the database from the context information we pass into the agent (using `langhraph.runtime.get_runtime()` function).

In [3]:
# let's define our tools as usual
from langchain.tools import tool
from langgraph.runtime import get_runtime


@tool
def execute_sql(query: str) -> str:
    """execute query provided by user
    Args:
        query (str): SQL query to execute
    Returns:
        str: result of the query execution or error message
    """
    # get instance of db in context
    db: SQLDatabase = get_runtime().context["db"]
    try:
        result = db.run(query)
    except Exception as e:
        return f"Error occurred while executing SQL query: {e}"
    return str(result)

**Step 3**

Next, we define the system prompt for our agent. Notice here that we have inserted a placeholder `{database_schema}` which we will _populate_ at runtime _before_ the agent sees the system prompt. We will discuss later how this can be accomplished.

In [4]:
# define our system prompt - modified here to pass in the database schema
SYSTEM_PROMPT = """You are a careful SQLite Analyst.

Rules:
- Always think step-by-step
- When you need data, call the tool 'execute_sql' with ONE select query
- Read-only only; NO INSERT/UPDATE/DELETE/DROP/CREATE/REPLACE/TRUNCATE/ALTER
- Limit to 5 rows at the output, unless the user explicitly asks for more
- If the tool returns "Error:", revise the SQL and try again
- Prefer explicit column list, avoid SELECT *
- Here is the database schema you can refer to to generate the SQL
{database_schema}
"""

**Step 4**

Here we modify the system prompt before it makes its way to the LLM in our agent. In the previous cell, we have inserted a placeholder `{database_schema}`, which we will _replace_ (or more precisely format) to insert the database schema. Our LLM can then refer to this schema to generate more accurate SQL queries faster. This technique is called **dynamic prompts**.

### Dynamic Prompts
Dynamic prompts are a core context engineering pattern - they adapt what you tell the model based on the current conversation state. LangChain provides the `@dynamic_prompt` to dynamically generate system prompts for the model. 

This is a convenience decorator that creates middleware using `wrap_model_call` (see image in the first cell of this notebook) _specifically_ for dynamic prompt generation. The decorated function should return a string that will be set as the system prompt for the model request. Under the hood, the LangChain runtime will call this function _before_ it makes the LLM call, so anything going into the LLM can be modified here. You have access to the `ModelRequest`, which gives you access to the _raw_ system prompt, the runtime context etc. See [ModelRequest API](https://reference.langchain.com/python/langchain/agents/middleware/types/ModelRequest) for details.

We format the globally defined SYSTEM_PROMPT with the database schema information, which we get from our runtime context an return the formatted system prompt.

In [5]:
# let's build a dynamic system prompt (i.e. we'll replace {database_schema}
# with the actual values extracted from DB at runtime
from langchain.agents.middleware.types import ModelRequest, dynamic_prompt


@dynamic_prompt
def dynamic_system_prompt(request: ModelRequest) -> str:
    # here we will format SYSTEM_PROMPT with actual value of db schema
    print("------ dynamic_system_prompt() called ---------")
    # grab the database schemd from the runtime context
    db_schema = request.runtime.context["db_schema"]
    return SYSTEM_PROMPT.format(database_schema=db_schema)

**Step 5**

Next we create our agent (we'll use the OpenAI `gpt-5-mini` model) using the `langchain.agents.create_agent()` function as usual. By now you must be familiar with most of the parameters to this function.

The `middleware` parameter is new - this parameter take a list of functions, which are esentially hooks into the lifecycle of an agent. For now, we are passing is just the `dynamic_system_prompt` hook we defined above.

In [6]:
# define our agent
from langchain.agents import create_agent
from typing import List

agent = create_agent(
    model="openai:gpt-5-mini",
    tools=[execute_sql],
    context_schema=RuntimeContext,
    response_format=Customers,
    middleware=[dynamic_system_prompt],
)

**Step 6**
Now let's query our database for customer information - in his example, we restrict ourselves to the customer table only! We stream our results as before - nothing new here!

In [7]:
user_query = "List all my customers living in the city of London or Paris"

for step in agent.stream(
    {"messages": [{"role": "user", "content": user_query}]},
    context=RuntimeContext(db=db, db_schema=schema),
    stream_mode="values",
):
    step["messages"][-1].pretty_print()

================================ Human Message =================================

List all my customers living in the city of London or Paris
------ dynamic_system_prompt() called ---------
================================== Ai Message ==================================
Tool Calls:
  execute_sql (call_atzl8nRGUHcYgGfBRkOMdXIP)
 Call ID: call_atzl8nRGUHcYgGfBRkOMdXIP
  Args:
    query: SELECT CustomerId, FirstName, LastName, Address, City, State, Country, Phone, Email FROM customers WHERE City IN ('London','Paris') LIMIT 5;
================================= Tool Message =================================
Name: execute_sql

[(39, 'Camille', 'Bernard', '4, Rue Milton', 'Paris', None, 'France', '+33 01 49 70 65 65', 'camille.bernard@yahoo.fr'), (40, 'Dominique', 'Lefebvre', '8, Rue Hanovre', 'Paris', None, 'France', '+33 01 47 42 71 71', 'dominiquelefebvre@gmail.com'), (52, 'Emma', 'Jones', '202 Hoxton Street', 'London', None, 'United Kingdom', '+44 020 7707 0707', 'emma_jones@hotmail.com')

### Filtering out PII information

Notice that I have received neatly parsed list of dicts - one for each row of data returned by the SQL query. The fields are parsed per the definition provided in the `response_format=Customers` parameter to the agent.

However, PII elements such as phone number and email are _clearly_ visible in the output. The objective of this example is to filter them out - more specifically REDACT then. Guess what? LangChain's middleware comes to our rescue again! Specifically the `PIIMiddleware` class.

LangChain provides a `PIIMiddleware` class in the `langchain.agents.middleware` package. It also provides some pre-defined filters for PII elements such as `email`, `credit cards` etc. For the full list see [API reference](https://reference.langchain.com/python/langchain/agents/middleware/pii/PIIMiddleware?_gl=1*q6e8ks*_gcl_au*MzAzOTE5NjkxLjE3NzQzNTU0MjM.*_ga*MzM4MDk1Mjk2LjE3NTE0NjQ5MDU.*_ga_47WX3HKKY2*czE3NzU2NDAxNjEkbzUxJGcxJHQxNzc1NjQzMzY5JGo1NSRsMCRoMA..)

Not all PII elements are pre-defined. For example, there is no pre-defined filter for phone numbers or names. However, we can define custom PII filters using reg-expressins for phone numbers. We'll address names a bit later.

Following cell shows how to use pre-defined `PIIMiddleware` for email and a custom `PIIMiddleware` for phone numbers. 
* Notice that in both these calls, we ask the middleware to REDACT (`strategy="redact"`) the information - instead of the field value, it will be displayed as "REDACTED_FieldName" in output. 
* The `apply_to_input=True` means _check_ (or fire) before model is called (this is default behavior).
There is also a `apply_to_output=True|False` parameter (False by default) that we have not used. Setting it to `True` will fire this middleware after LLM generates it's output. 
* The `apply_to_tool_results=True` means, apply this filter after the tool call (`execute_sql`) and _before_ results of the tool call go to the LLM.

LangChain provides some pre-defined filters -> `pii_type=Literal['email', 'credit_card', 'ip', 'mac_address', 'url']`.

We have also defined a custom `PIIMiddlware` filter for the phone number. We have provided one large reg-expression to check value. This works on the "phone" field and provided a `detector=PHONE_REGEX` to detect phone number pattern and REDACT the value, should the value in the phone field match the reg-ex pattern defined.

In [8]:
from langchain.agents.middleware import PIIMiddleware

# email detector - in-built
email_detector = PIIMiddleware(
    pii_type="email",
    strategy="redact",
    apply_to_input=False,
    apply_to_tool_results=True,
)

# phone number - no in-built detector, so we use a regex

# Comprehensive pattern for North America, LATAM, Europe, Asia, and Africa
PHONE_REGEX = (
    r"(?:"
    # 1. International E.164 style (Starts with +)
    # Covers Europe, Asia, Africa, LATAM (+44, +91, +234, +49, etc.)
    r"\+\d{1,3}[\s\-\.]?\(?\d{1,4}\)?(?:[\s\-\.]\d{1,5}){1,4}"
    r"|"
    # 2. North American Numbering Plan (US/Canada)
    # Matches (555) 123-4567, 555-123-4567, 555.123.4567
    r"\(?\d{3}\)?[\s\-\.]\d{3}[\s\-\.]\d{4}"
    r"|"
    # 3. Generic Local Long Form (8-13 digits with separators)
    # Catches local formats in China, Japan, and parts of Europe
    r"(?:\b|(?<=\s))\d{2,4}[\s\-\.]\d{3,4}[\s\-\.]\d{3,4}\b"
    r")"
)


phone_detector = PIIMiddleware(
    "phone",
    # Reg-ex to detect
    detector=PHONE_REGEX,
    strategy="redact",
    apply_to_input=False,
    apply_to_tool_results=True,
)

Here is how we _attach_ the pre-defined and custom `PIIMiddleware` to our agent. Following is the definition of our `pii_agent`, which is _powered_ with `PIIMiddleware` objects to filter our email and phone number in addition to our `dynamic_system_prompt` middleware defined to specifically modify the system prompt.

In [9]:
# now let's modify our agent definition
from langchain.agents import create_agent
from typing import List

pii_agent = create_agent(
    model="openai:gpt-5-mini",
    tools=[execute_sql],
    context_schema=RuntimeContext,
    response_format=Customers,
    middleware=[
        dynamic_system_prompt,
        email_detector,
        phone_detector,
    ],
)

Next, we run the same query as before, but now with our `pii_agent`. 

In [10]:
# now let's ask the same query to the pii_agent
user_query = "List all my customers living in the city of London or Paris"

for step in pii_agent.stream(
    {"messages": [{"role": "user", "content": user_query}]},
    context=RuntimeContext(db=db, db_schema=schema),
    stream_mode="values",
):
    step["messages"][-1].pretty_print()

================================ Human Message =================================

List all my customers living in the city of London or Paris
------ dynamic_system_prompt() called ---------
================================== Ai Message ==================================
Tool Calls:
  execute_sql (call_ifkKXx7S3TDZfXsVONWB9sKc)
 Call ID: call_ifkKXx7S3TDZfXsVONWB9sKc
  Args:
    query: SELECT CustomerId, FirstName, LastName, Company, Address, City, State, Country, PostalCode, Phone, Fax, Email
FROM customers
WHERE City IN ('London', 'Paris')
LIMIT 5;
================================= Tool Message =================================
Name: execute_sql

[(39, 'Camille', 'Bernard', None, '4, Rue Milton', 'Paris', None, 'France', '75009', '+33 01 49 70 65 65', None, 'camille.bernard@yahoo.fr'), (40, 'Dominique', 'Lefebvre', None, '8, Rue Hanovre', 'Paris', None, 'France', '75002', '+33 01 47 42 71 71', None, 'dominiquelefebvre@gmail.com'), (52, 'Emma', 'Jones', None, '202 Hoxton Street', 'Lond

Notice that the structure of the output is the same as before, but the contents of the phone & email fields have been REDACTED - displaye [REDACTED_PHONE] and [REDACTED_EMAIL] where the values were displayed previously.


### Redacting the First and Last Names
By adding `PIIMiddleware` filters for email & phone, we saw that the agent is able to redact this information. Since these fields follow some sort of pattern, we can detect them early and filter out the information. Doing this with free-text fields such as names & addresses is a challenge. 

To address such filtering, LangChain provides a `@after_model` decorator, which you can apply to a custom function that will be called _after_ model generated output but _before_ this output is sent to the user. Since our output is structured, it will be able in the response['structured_output`] field (as we saw in the [LLM with Structured Output](07_structured_output.ipynb) example). In this function we iterate over the fields and explicitly set values of `firstName` and `lastName` to `[REDACTED]`.

In [14]:
from langchain.agents.middleware.types import after_model
from langchain.agents.middleware import AgentState

# from langchain.agents.middleware.types import Runtime
from langgraph.runtime import Runtime

REDACTED = "[REDACTED]"


@after_model
def redact_name_fields(state: AgentState, runtime: Runtime) -> dict | None:
    structured = state.get("structured_response")

    # if no output from model, return nothing!
    if not structured:
        return None

    # this call replaces values of firstName and lastName with [REDACTED]
    redacted_customers = [
        c.model_copy(update={"firstName": REDACTED, "lastName": REDACTED})
        for c in structured.customers
    ]
    return {"structured_response": Customers(customers=redacted_customers)}

In [15]:
# define our agent, with new @after_model middleware
from langchain.agents import create_agent

pii_agent2 = create_agent(
    model="openai:gpt-5-mini",
    tools=[execute_sql],
    context_schema=RuntimeContext,
    response_format=Customers,
    middleware=[
        dynamic_system_prompt,
        email_detector,
        phone_detector,
        redact_name_fields,
    ],
)

In [ ]:
# execute the same query as before
user_query = "List all my customers living in the city of London or Paris"

final_step = None
for step in pii_agent2.stream(
    {"messages": [{"role": "user", "content": user_query}]},
    context=RuntimeContext(db=db, db_schema=schema),
    stream_mode="values",
):
    step["messages"][-1].pretty_print()
    final_step = step

# The @after_model middleware updates structured_response in the agent state,
# NOT the raw AI message text — so we must read structured_response to see
# the redacted firstName / lastName values.
if final_step and "structured_response" in final_step:
    print(
        "\n--- Final structured output (names redacted by @after_model middleware) ---"
    )
    print(final_step["structured_response"].model_dump_json(indent=2))

================================ Human Message =================================

List all my customers living in the city of London or Paris
------ dynamic_system_prompt() called ---------
================================== Ai Message ==================================
Tool Calls:
  execute_sql (call_Lte2nKi18Dn5P0s3ZEsFLQlU)
 Call ID: call_Lte2nKi18Dn5P0s3ZEsFLQlU
  Args:
    query: SELECT CustomerId, FirstName, LastName, Company, Address, City, State, Country, PostalCode, Phone, Fax, Email
FROM customers
WHERE City IN ('London','Paris')
ORDER BY LastName, FirstName
LIMIT 5;
================================= Tool Message =================================
Name: execute_sql

[(39, 'Camille', 'Bernard', None, '4, Rue Milton', 'Paris', None, 'France', '75009', '+33 01 49 70 65 65', None, 'camille.bernard@yahoo.fr'), (53, 'Phil', 'Hughes', None, '113 Lupus St', 'London', None, 'United Kingdom', 'SW1V 3EN', '+44 020 7976 5722', None, 'phil.hughes@gmail.com'), (52, 'Emma', 'Jones', None, '2